# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library and the Croissant schema. All entities in the dataset—such as record sets, fields, and columns—are referenced by their `@id` for clarity and reproducibility.

## Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Install the mlcroissant library if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # This is a CroissantMetadata object (not dict)

print(f"Dataset: {meta.name}\nDescription: {meta.description}")
print(f"Published: {meta.date_published}")
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List the record sets and fields available in the dataset
print("Record Sets in Dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id} | name: {rs.name} | description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - @id: {col.id} | name: {col.name} | path: {col.path}")
    print()

## 3. Data Extraction
Load records from a specific record set into a pandas DataFrame for analysis. Reference the record set and field `@id`s from the overview above.

In [ ]:
# Choose main record set (by @id) for analysis
# Example: Let's use the first available record set for demonstration.
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet: {record_set_id}")

# Show available columns in the first record set
if record_sets:
    example_record_set_id = record_sets[0]
    print("Columns in the first record set:")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping by selecting numeric and grouping fields via their `@id`.

In [ ]:
# Select main record set and fields for demonstration
main_rs_id = record_sets[0]
df = dataframes[main_rs_id]

# Choose a numeric field and a grouping field via @id (change as appropriate for your dataset)
# We'll first display all columns to let the user pick if working interactively.
print("Available columns for EDA:", df.columns.tolist())

# Example: Use 'phq9_total' if present (commonly total score for PHQ-9), group by 'gender' if present
possible_numeric_fields = ['phq9_total', 'gad7_total', 'psq_total', 'age']
possible_group_fields = ['gender', 'marital_status', 'level_of_education']

numeric_field = None
for col in possible_numeric_fields:
    if col in df.columns:
        numeric_field = col
        break

group_field = None
for col in possible_group_fields:
    if col in df.columns:
        group_field = col
        break

if numeric_field is None or group_field is None:
    print("Could not find appropriate numeric/grouping fields in the imported DataFrame. Please check column names.")
else:
    # Example: Filter for high PHQ-9 scores
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a demographic field and calculate mean
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

if group_field is not None:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
This notebook introduced the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, and demonstrated how to load the dataset, explore structure via record and field `@id`s, extract and process records, and perform exploratory data analysis and visualization using `mlcroissant`.

- The dataset provides demographic and mental health indicator variables, with standardized IDs for reproducibility.
- Data exploration identified key numerical and categorical fields for further statistical or ML research.

Remember to always reference record sets, fields, and columns by their `@id` for programmatic clarity and reproducibility.